# Racial Biases in Police Killings in the United States
by Ishan Gohel

Question: What racial biases play into police killings throughout the United States?

In [ ]:
import kagglehub
import altair as alt
import pandas as pd

source = "https://github.com/fivethirtyeight/data/raw/refs/heads/master/police-killings/police_killings.csv"

df = pd.read_csv(source,encoding="latin-1")

With the rise in popularity of movements like Black Lives Matter and protests following the 2020 murder of George Floyd, my exploratory analysis in this dataset aims to find a trend of racial biases in police killings in the United States. This dataset explores the 2015 dataset from FiveThirtyEight exploring data from police killings throughout the US from January to June 2015. It has different columns displaying Race/Ethnicity, Age, and other demographic factors.

In [ ]:
# counts percentage of police killings by race
deaths_per_race = (df["raceethnicity"].value_counts(normalize=True)*100).round(2)

# pulled demographic data from the US Census Bureau: https://data.census.gov/table/ACSDP5Y2015.DP05
demographics_us_2015 = {"White":62.3,"Black":12.3,"Hispanic/Latino":17.1,"Asian":5.1,"Native American":0.7}

# created two rows of data comparing percentage of police killings to percentage of US population by race
rows = []
for race, percentage in demographics_us_2015.items():
  rows.append({"race": race, "metric": "Percentage of police killings", "percent": float(deaths_per_race.get(race, 0))})
  rows.append({"race": race, "metric": "Percentage of US population", "percent": percentage})

data = pd.DataFrame(rows)

# sorted data from highest to lowest percentage of police killings
sort_order = (data[data["metric"] == "Percentage of police killings"].sort_values("percent", ascending=False)["race"].tolist())

# created a hover feature to see data when hovering over bar chart
hover = alt.selection_point(on="mouseover", fields=["race"], empty=True)

# created bar chart
percentage_comparison = (alt.Chart(data).mark_bar().encode(
      x=alt.X("race:N", sort=sort_order, title="Race/Ethnicity",axis=alt.Axis(labelAngle=0)), # sets x-axis to Race/Ethnicity
      y=alt.Y("percent:Q", title="Percentage"), # sets y-axis to Percentage
      xOffset=alt.XOffset("metric:N"), # places both bars next to each other instead of on top, making the data more readable
      color=alt.Color("metric:N",scale=alt.Scale(domain=["Percentage of police killings", "Percentage of US population"],range=["#40798C", "#CFE0C3"])),
      opacity=alt.condition(hover, alt.value(1.0), alt.value(0.35)),
      tooltip=["race:N", "metric:N", alt.Tooltip("percent:Q", format=".1f")],
    ).add_params(hover).properties(width=500, height=250) # created a tooltip showing race and percentage of police killings
)

percentage_comparison

alt.Chart(...)

Despite making up around 13% of the overall US population in 2015, Black people are significantly overrepresented in this graph of overall police killings in the US, making up about 28% of all police killings in the dataset. Native American people in the United States are also overrepresented in their percentage of overall police killings, making up 0.9% of police killings despite being only 0.7% of the total US population.

In [ ]:
# clean up the ages
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# dropna of ages so there is no missing data
plot = df.dropna(subset=["age"]).copy()
plot = plot[plot["raceethnicity"] != "Unknown"]

# color palette
color_palette = {
    "White": "#98C300",
    "Black": "#62BFED",
    "Hispanic/Latino": "#F15BB5",
    "Asian/Pacific Islander": "#FF521B",
    "Native American": "#433E3F",
}

# creating a brush so that highlighting a region in the age range translates to the below graph
brush = alt.selection_interval()

# histogram of victim age
histogram_of_age = (alt.Chart(plot).mark_bar().encode(
        x=alt.X("age:Q",bin=alt.Bin(maxbins=25),title="Age"), # creates X-axis of age
        y=alt.Y("count():Q",title="Number of Police Killings"), # creates Y-axis of number of police killings
        color=alt.condition(brush,alt.value("#40798C"),alt.value("#CCCCCC")), # sets colors of graph to turquoise and grey (if unselected)
        tooltip=[alt.Tooltip("count():Q", title="Killings"),alt.Tooltip("age:Q", bin=alt.Bin(maxbins=25), title="Age Range"),],) # creates the tooltip when hovering
    .add_params(brush).properties(width=500, height=250)
)

# racial composition in selected age group
victim_racial_composition = (alt.Chart(plot).mark_bar().encode(
        x=alt.X("count():Q", title="Number of Police Killings"), # creates x-axis of police killings
        y=alt.Y("raceethnicity:N",sort="-x",title="Race/Ethnicity"), # y-axis of race/ethnicity
        color=alt.Color("raceethnicity:N",scale=alt.Scale(domain=list(color_palette.keys()),range=list(color_palette.values()))), # sets colors
        tooltip=[alt.Tooltip("raceethnicity:N", title="Race"),alt.Tooltip("count():Q", title="Killings in Selection"),],) # creates tooltip displaying race and killings in the specified selection
    .transform_filter(brush)
    .properties(width=500, height=250,title="Racial Demographics of Age Range",)
)

# displays both charts

histogram_of_age & victim_racial_composition

alt.VConcatChart(...)

When selecting younger age groups, like from 20-25 years old, Black people make up the highest amount of police killings, followed by Hispanic/Latino. However, when selecting older age ranges, like from 50-55, White people make up the highest amount of police killings. When it comes to younger people, which accounts for most of police killings, Black and Hispanic people are killed at disproportional rates, whereas older, middle-age white people are killed at higher rates than other demographics.

There exists a clear racial bias in the number of police killings, and young Black and Hispanic people are at the highest risk of their lives ending at the hands of police.

For the first graph, I chose a green gradient. I chose the colors in the second plot so that graphs that weren't selected were easily distingusihable from and less opaque than graphs that were chosen. I chose the color palette in the second graph so that there is a heavy visible difference in each demographic.